In [18]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [19]:
from pathlib import Path

folder = Path("data/external")

print("Folder exists:", folder.exists())

if folder.exists():
    print("\nFiles inside:")
    for f in folder.iterdir():
        print(f.name)

Folder exists: True

Files inside:
PSA_Parallel_Dataset.xlsx


In [20]:
import pandas as pd

from SRC.excel_manager import ExcelManager

excel = ExcelManager()

print("Workbook loaded successfully.")

Workbook loaded successfully.


In [21]:
sample_sentences = [

    {
        "Domain":"Health",
        "English":"Wash your hands regularly with soap and clean water.",
        "Source":"WHO"
    },

    {
        "Domain":"Health",
        "English":"Cover your mouth when coughing or sneezing.",
        "Source":"WHO"
    },

    {
        "Domain":"Road Safety",
        "English":"Always wear your seat belt while driving.",
        "Source":"NTSA"
    }

]

In [22]:
for i, row in enumerate(sample_sentences, start=1):

    excel.add_record(

        psa_id=f"PSA{i:06d}",

        domain=row["Domain"],

        english=row["English"],

        source=row["Source"]

    )

excel.save()

print("Sample records added successfully.")

Sample records added successfully.


In [23]:
import pandas as pd

columns = [
    "PSA_ID",
    "Domain",
    "English",
    "Kiswahili",
    "Dholuo",
    "Source",
    "Date",
    "Metadata",
    "Status"
]

master_df = pd.DataFrame(columns=columns)

master_df.head()

,PSA_ID,Domain,English,Kiswahili,Dholuo,Source,Date,Metadata,Status


In [24]:
from datetime import datetime

def add_record(df,
               domain,
               english,
               source,
               metadata="scraped"):

    psa_id = f"PSA{len(df)+1:06d}"

    new_row = {

        "PSA_ID": psa_id,

        "Domain": domain,

        "English": english,

        "Kiswahili": "",

        "Dholuo": "",

        "Source": source,

        "Date": datetime.today().strftime("%Y-%m-%d"),

        "Metadata": metadata,

        "Status": "Pending"

    }

    return pd.concat(
        [df, pd.DataFrame([new_row])],
        ignore_index=True
    )

In [25]:
master_df = add_record(
    master_df,
    "Health",
    "Wash your hands regularly with soap and clean water.",
    "WHO"
)

master_df = add_record(
    master_df,
    "Road Safety",
    "Always wear a seat belt while driving.",
    "NTSA"
)

master_df

,PSA_ID,Domain,English,Kiswahili,Dholuo,Source,Date,Metadata,Status
0,PSA000001,Health,Wash your hands regularly with soap and clean ...,,,WHO,2026-07-22,scraped,Pending
1,PSA000002,Road Safety,Always wear a seat belt while driving.,,,NTSA,2026-07-22,scraped,Pending


In [26]:
master_df.to_excel(
    "data/external/PSA_Parallel_Dataset.xlsx",
    sheet_name="Master_Dataset",
    index=False
)

print("Workbook updated successfully!")

Workbook updated successfully!


In [27]:
from SRC.psa_collector import PSACollector

collector = PSACollector()

print("Collector loaded.")

Collector loaded.


In [28]:
url = "https://www.who.int/news-room/questions-and-answers/item/coronavirus-disease-covid-19"

paragraphs = collector.collect(url)

print("Paragraphs collected:", len(paragraphs))

paragraphs[:10]

Paragraphs collected: 57


['WHO is continuously monitoring and responding to this pandemic. This questions and answers page will be updated as more is known about COVID-19, how it spreads and how it is affecting people worldwide. For more information, regularly check the WHO coronavirus pages. https://www.who.int/covid-19',
 'COVID-19 is the disease caused by a coronavirus called\r\nSARS-CoV-2. \xa0WHO first learned of this new virus on 31 December 2019,\r\nfollowing a report of a cluster of cases of so-called viral pneumonia in Wuhan,\r\nPeople’s Republic of China.',
 'COVID-19 is the disease caused by a coronavirus called\r\nSARS-CoV-2. \xa0WHO first learned of this new virus on 31 December 2019,\r\nfollowing a report of a cluster of cases of so-called viral pneumonia in Wuhan,\r\nPeople’s Republic of China.',
 'The most common symptoms of COVID-19 are fever chills sore throat. Other symptoms that are less common and may affect some patients include: muscle aches severe fatigue or tiredness runny or blocked n

In [29]:
for paragraph in paragraphs:

    master_df = add_record(
        master_df,
        domain="Health",
        english=paragraph,
        source="WHO"
    )

master_df.head()

print("Total records:", len(master_df))



Total records: 59


In [30]:
print("Dataset saved.")

Dataset saved.


In [31]:
from SRC.sources import SOURCES
from SRC.psa_collector import PSACollector

collector = PSACollector()

print(f"Loaded {len(SOURCES)} sources.")

Loaded 8 sources.


In [32]:
master_df = master_df.iloc[0:0]

for source in SOURCES:

    print(f"\nCollecting from {source['source']}...")

    paragraphs = collector.collect(source["url"])

    print(f"Collected {len(paragraphs)} paragraphs.")

    for paragraph in paragraphs:

        master_df = add_record(
            master_df,
            domain=source["domain"],
            english=paragraph,
            source=source["source"]
        )


Collected 57 paragraphs.

Collected 5 paragraphs.

Failed: https://www.unicef.org/parenting
403 Client Error: Forbidden for url: https://www.unicef.org/parenting
Collected 0 paragraphs.

Failed: https://ntsa.go.ke
HTTPSConnectionPool(host='ntsa.go.ke', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
Collected 0 paragraphs.

Failed: https://www.nationalpolice.go.ke
HTTPSConnectionPool(host='www.nationalpolice.go.ke', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010)')))
Collected 0 paragraphs.

Failed: https://kilimo.go.ke
HTTPSConnectionPool(host='kilimo.go.ke', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFI

In [33]:
before = len(master_df)

master_df = master_df.drop_duplicates(
    subset=["English"]
)

master_df = master_df.reset_index(drop=True)

after = len(master_df)

print(f"Removed {before-after} duplicates")
print(f"Remaining: {after}")

Removed 6 duplicates
Remaining: 70


In [34]:
master_df.to_excel(
    "data/external/PSA_Parallel_Dataset.xlsx",
    index=False
)

master_df.to_csv(
    "data/raw/english_psa.csv",
    index=False
)

print("Dataset exported successfully!")

Dataset exported successfully!
